# 10 · Data wrangling checkpoint: a reproducible survey

**Module:** 2 · **Time:** 55 minutes · **Prerequisites:** Modules 1–2, including merging and reshaping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/10_wrangling_project.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F10_wrangling_project.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** Badges target `main` and public access is not yet available; this repository is private. This checkpoint is intentionally less guided than ordinary lessons.

## Learning goals

- **Audit** a deterministic fictional survey for missing codes and duplicate identifiers.
- **Combine** person and household data, then reshape repeated wage columns.
- **Verify** a reproducible, analysis-ready Parquet export without releasing private data.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(20261101)
regions = ["North", "SOUTH", "east", "West"]
statuses = ["employed", "unemployed", "not in labor force"]
print("Generator seed:", 20261101)

## Why this matters

This checkpoint uses the same fixed seed and row-construction rules as the repository's synthetic survey generator. **All 90 observations are fictional**, not estimates for actual households. Your task is to inspect the raw inputs, clean them, verify the merge relationship, reshape the two wage waves, and demonstrate that an exported dataset can be read back.

## Reproduce the synthetic inputs

The row factory below follows the deterministic example generator. It includes mixed-date formats, string wages, and survey weights. Always use a seed when generating practice data so others can reproduce your exercise.

In [ ]:
def make_response(i):
    wage = int(rng.integers(12000, 88000))
    day = f"{i % 27 + 1:02d}"
    date = f"2024-03-{day}" if i % 2 == 0 else f"03/{day}/2024"
    return {
        "person_id": 1001 + i,
        "household_id": f"H{i // 3 + 1:03d}",
        "region": regions[i % len(regions)],
        "employment": statuses[i % len(statuses)],
        "interview_date": date,
        "wage_2023": str(wage),
        "wage_2024": str(round(wage * float(rng.uniform(0.9, 1.2)))),
        "survey_weight": round(float(rng.uniform(0.6, 2.4)), 3),
    }

In [ ]:
rows = [make_response(i) for i in range(90)]
rows[0]["wage_2023"] = "-99"
rows[1]["wage_2024"] = "not reported"
rows[2]["region"] = " north "
rows[3]["wage_2024"] = "9999999"
rows[5]["person_id"] = rows[4]["person_id"]
survey = pd.DataFrame(rows)
print("Raw rows:", len(survey))
print("Duplicated person IDs:", survey["person_id"].duplicated().sum())

In [ ]:
household_ids = sorted(survey["household_id"].unique())
households = pd.DataFrame({
    "household_id": household_ids,
    "housing_type": ["rent" if i % 2 else "own" for i in range(len(household_ids))],
    "urban": [int(i % 3 != 0) for i in range(len(household_ids))],
})
print("Household rows:", len(households))
print("Household key unique:", households["household_id"].is_unique)

## Preflight audit

One duplicate person ID is intentional. It does not authorize dropping a row: you must decide whether you have a duplicate observation or an identifier problem. The survey rows themselves are the unit in the reshape exercise, so keep a separate unique `record_id`.

In [ ]:
print(survey[["person_id", "household_id", "wage_2023", "wage_2024"]].head(6))
print("Weights all positive:", survey["survey_weight"].gt(0).all())
print("Household records match lookup:", survey["household_id"].isin(household_ids).all())

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Missing code | `s.mask(s.eq(-99))` | `replace wage = . if wage == -99` | `na_if(s, -99)` |
| Merge lookup | `merge(..., validate='many_to_one')` | `merge m:1 household_id using ...` | `left_join(..., relationship='many-to-one')` |
| Wide to long | `melt(...)` | `reshape long` | `pivot_longer(...)` |
| Export Parquet | `to_parquet(path)` | via compatible import/export tools | `arrow::write_parquet(...)` |

An analysis-ready dataset should preserve provenance, keys, types, and a reproducible cleaning procedure.

## Check your Parquet engine

The locked course environment contains pyarrow, while a minimal local sandbox might not. First inspect availability; the export check itself remains part of Exercise 5.

In [ ]:
from importlib.util import find_spec

print("Parquet engine available:", find_spec("pyarrow") is not None)

## Try it

Complete these five tasks independently. Answer cells start blank in the learner notebook; each check offers a hint without stopping execution. **Do not silently drop observations to make a check pass.**

### ★ Exercise 1 — Clean regions and wages

Create `cleaned_survey` as a new DataFrame with `region_clean`, `wage23`, and `wage24`. Normalize region text; parse numeric strings; treat -99 and 2024 wages above 500,000 as missing. Keep the raw columns and all rows.

In [ ]:
cleaned_survey = None

In [ ]:
expected23 = pd.to_numeric(survey["wage_2023"], errors="coerce").mask(lambda s: s.eq(-99))
expected24 = pd.to_numeric(survey["wage_2024"], errors="coerce").mask(lambda s: s.gt(500_000))
if (isinstance(cleaned_survey, pd.DataFrame) and len(cleaned_survey) == len(survey)
    and all(c in cleaned_survey for c in ["region_clean", "wage23", "wage24"])
    and cleaned_survey["wage23"].equals(expected23)
    and cleaned_survey["wage24"].equals(expected24)
    and cleaned_survey["region_clean"].equals(survey["region"].str.strip().str.lower())):
    print("✓ Cleaning audit passed")
else:
    print("Hint: preserve raw rows and build three new clean columns.")

### ★★ Exercise 2 — Investigate duplicate person IDs

Create `duplicate_flags` as a Boolean Series marking every member of a duplicate person-ID group. Preserve the suspicious observations for review.

In [ ]:
duplicate_flags = None

In [ ]:
expected_flags = survey.duplicated("person_id", keep=False)
if isinstance(duplicate_flags, pd.Series) and duplicate_flags.equals(expected_flags):
    print("✓ Duplicate IDs flagged")
else:
    print("Hint: use duplicated('person_id', keep=False).")

### ★★ Exercise 3 — Merge the household lookup

Create `joined_survey` with a left many-to-one merge on `household_id`. Keep every person row and include the household variables.

In [ ]:
joined_survey = None

In [ ]:
expected_join = survey.merge(households, how="left", on="household_id", validate="many_to_one")
if isinstance(joined_survey, pd.DataFrame) and joined_survey.equals(expected_join):
    print("✓ Household merge checked")
else:
    print("Hint: use a left merge with validate='many_to_one'.")

### ★★★ Exercise 4 — Reshape the two waves

Create `long_survey`: start from `survey` with a unique `record_id` from the row index, then melt `wage_2023` and `wage_2024` into columns named `wave` and `wage_text`. Retain `record_id` and `household_id` as identifiers.

In [ ]:
long_survey = None

In [ ]:
expected_long = survey.reset_index(names="record_id").melt(
    id_vars=["record_id", "household_id"],
    value_vars=["wage_2023", "wage_2024"], var_name="wave", value_name="wage_text"
)
if isinstance(long_survey, pd.DataFrame) and long_survey.equals(expected_long):
    print("✓ Two-wave reshape checked")
else:
    print("Hint: use reset_index(names='record_id') and melt two wage columns.")

### ★★★ Exercise 5 — Analysis-ready Parquet round trip

Combine your cleaned person-level columns with the household lookup, reshape the two clean numeric wages to a long record-year table, and export that **analysis-ready** result to a temporary Parquet file. Set `parquet_roundtrip_ok` to True only when the reloaded table equals the exported one. A local environment without pyarrow cannot verify the export, but the locked CI environment does.

In [ ]:
parquet_roundtrip_ok = None

In [ ]:
if parquet_roundtrip_ok == "engine-unavailable" and find_spec("pyarrow") is None:
    print("✓ Export check deferred: pyarrow unavailable; locked CI will verify.")
elif parquet_roundtrip_ok is True:
    print("✓ Analysis-ready Parquet export verified")
else:
    print("Hint: create the cleaned merged long table and verify a Parquet round trip.")

## Common errors

- Using `person_id` as a unique reshape key fails because this synthetic survey deliberately repeats it once. Introduce and document a true record key.
- Converting `-99` directly to a wage leaves an invalid observation; check source missing codes first.
- A merge without `validate='many_to_one'` can silently multiply rows when lookup keys repeat.
- A Parquet engine such as `pyarrow` is required; install the pinned environment rather than guessing versions.

## Recap

- You audited a deterministic, entirely fictional household survey.
- You built new variables without overwriting raw evidence.
- You checked duplicate keys and a many-to-one household merge.
- You practiced wide-to-long transformation and a reproducible export.

## Self-assessment

- [ ] I can explain why `person_id` is not uniquely identifying this dataset.
- [ ] I can reproduce the same 90-row input from the documented random seed.
- [ ] I can distinguish an omitted wage from a valid zero.
- [ ] I can validate the merge relationship and ensure no rows were multiplied.
- [ ] I can reshape to one row per record-year and verify the Parquet round trip.

## Next steps

Continue to [Module 3: descriptive tables and visualization](../03_describe_visualize/README.md) (planned). For production research, separate generation, cleaning, analysis, and output into a reproducible pipeline; see [pandas' I/O documentation](https://pandas.pydata.org/docs/user_guide/io.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).